# J.U.D.E. Tier 1 Intent Router — Training, Calibration & INT8 ONNX Export (CL461)

This notebook trains and evaluates **Tier 1 (System 1)** of **J.U.D.E.** on Google Colab Pro (**T4 GPU** recommended):

1. **Idempotent GitHub Clone:** Pulls `https://github.com/Baymax005/JUDE.git` (`intents_dataset.csv` + `ood_test_utterances.csv`).
2. **Stratified 80 / 10 / 10 Split (`SEED=42`):** 864 train / 108 val / 108 test across 9 intents.
3. **Baseline (TF-IDF + Logistic Regression):** Tuned on validation Macro-F1 and evaluated on the identical test split.
4. **Fine-Tuned `distilbert-base-uncased`:** 6 epochs, per-epoch Loss & Macro-F1 curves (`loss_curve.png`), per-class Precision/Recall/F1, and `confusion_matrix.png`.
5. **Temperature Scaling & 3-Way Confidence Gate Validation:** Fits temperature `T` on validation NLL, reports ECE before/after scaling, evaluates 30 unseen OOD utterances against `>=0.80` (Direct) / `0.55–0.80` (Confirm) / `<0.55` (Escalate), and saves `confidence_histogram_ood.png`.
6. **INT8 ONNX Export & CPU Benchmark:** Exports `model_fp32.onnx`, quantizes to `model_quantized.onnx` (INT8), verifies 100% tokenizer parity with `tokenizers`, benchmarks FP32 PyTorch vs. INT8 ONNX on CPU (`fp32_vs_int8_benchmark.png`), and packages `tier1_router_bundle.zip` + `cl461_report_assets.zip`.

## 1. Environment Setup & Idempotent GitHub Clone

In [ ]:
!pip install -q onnx onnxruntime accelerate

import os
import random
import subprocess
from pathlib import Path

import numpy as np
import torch
import transformers
import onnxruntime as ort

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

REPO_DIR = Path("/content/JUDE")
REPO_URL = "https://github.com/Baymax005/JUDE.git"

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

OUT_ROOT = Path("/content/jude_outputs")
ROUTER_DIR = OUT_ROOT / "tier1_router"
CHARTS_DIR = OUT_ROOT / "report_charts"
ROUTER_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY (Switch runtime to T4 GPU!)"
print(f"GPU / Device     : {gpu_name}")
print(f"PyTorch          : {torch.__version__}")
print(f"Transformers     : {transformers.__version__}")
print(f"ONNX Runtime     : {ort.__version__}")
print(f"Repo Dir         : {REPO_DIR}")
print(f"Outputs Dir      : {OUT_ROOT}")

## 2. Load Datasets & Reproducible Stratified 80 / 10 / 10 Split

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(REPO_DIR / "colab" / "intents_dataset.csv")
ood_df = pd.read_csv(REPO_DIR / "colab" / "ood_test_utterances.csv")

assert list(df.columns) == ["text", "intent", "tier", "style"], f"Unexpected columns: {df.columns}"
assert list(ood_df.columns) == ["text", "category", "notes"], f"Unexpected OOD columns: {ood_df.columns}"
assert df.isna().sum().sum() == 0, "Null values found in intents_dataset.csv"
assert ood_df.isna().sum().sum() == 0, "Null values found in ood_test_utterances.csv"

id_lower = set(df["text"].str.strip().str.lower())
ood_lower = set(ood_df["text"].str.strip().str.lower())
assert len(id_lower) == len(df), "Duplicate texts found in intents_dataset.csv"
assert len(ood_lower) == len(ood_df), "Duplicate texts found in ood_test_utterances.csv"
assert id_lower.isdisjoint(ood_lower), "Overlap detected between ID and OOD datasets!"

LABELS = sorted(df["intent"].unique().tolist())
label2id = {label: idx for idx, label in enumerate(LABELS)}
id2label = {idx: label for idx, label in enumerate(LABELS)}
intent_to_tier = df.groupby("intent")["tier"].first().to_dict()

df["label"] = df["intent"].map(label2id)

train_df, temp_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Total In-Distribution : {len(df)} | Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}")
print(f"Total OOD Test Queries: {len(ood_df)} -> {ood_df['category'].value_counts().to_dict()}")
print("\nClass & Style Distribution:")
summary = pd.DataFrame({
    "tier": [intent_to_tier[l] for l in LABELS],
    "total": [int((df["intent"] == l).sum()) for l in LABELS],
    "train": [int((train_df["intent"] == l).sum()) for l in LABELS],
    "val": [int((val_df["intent"] == l).sum()) for l in LABELS],
    "test": [int((test_df["intent"] == l).sum()) for l in LABELS],
    "typed": [int(((df["intent"] == l) & (df["style"] == "typed")).sum()) for l in LABELS],
    "voice": [int(((df["intent"] == l) & (df["style"] == "voice")).sum()) for l in LABELS],
}, index=LABELS)
display(summary)

## 3. CL461 Baseline: TF-IDF + Logistic Regression (Same 80 / 10 / 10 Split)

In [ ]:
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)

best_c = None
best_val_f1 = -1.0
best_baseline = None

for c_val in [0.5, 1.0, 2.0, 5.0, 10.0]:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, lowercase=True)),
        ("clf", LogisticRegression(C=c_val, max_iter=2000, random_state=SEED)),
    ])
    pipe.fit(train_df["text"], train_df["label"])
    val_preds = pipe.predict(val_df["text"])
    _, _, val_f1, _ = precision_recall_fscore_support(
        val_df["label"], val_preds, average="macro", zero_division=0
    )
    print(f"TF-IDF + LogReg (C={c_val:<4}) -> Val Macro-F1: {val_f1:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_c = c_val
        best_baseline = pipe

baseline_test_preds = best_baseline.predict(test_df["text"])
b_acc = accuracy_score(test_df["label"], baseline_test_preds)
b_prec, b_rec, b_f1, _ = precision_recall_fscore_support(
    test_df["label"], baseline_test_preds, average="macro", zero_division=0
)
_, _, b_wf1, _ = precision_recall_fscore_support(
    test_df["label"], baseline_test_preds, average="weighted", zero_division=0
)

typed_mask = (test_df["style"] == "typed").to_numpy()
voice_mask = (test_df["style"] == "voice").to_numpy()
b_typed_acc = accuracy_score(test_df["label"][typed_mask], baseline_test_preds[typed_mask])
b_voice_acc = accuracy_score(test_df["label"][voice_mask], baseline_test_preds[voice_mask])

# Single-query latency benchmark
sample_texts = test_df["text"].tolist()
t0 = time.perf_counter()
for txt in sample_texts:
    best_baseline.predict([txt])
b_latency_ms = ((time.perf_counter() - t0) / len(sample_texts)) * 1000.0

baseline_metrics = {
    "model": f"TF-IDF + LogisticRegression (C={best_c})",
    "best_c": float(best_c),
    "val_macro_f1": float(best_val_f1),
    "test_accuracy": float(b_acc),
    "macro_precision": float(b_prec),
    "macro_recall": float(b_rec),
    "macro_f1": float(b_f1),
    "weighted_f1": float(b_wf1),
    "typed_accuracy": float(b_typed_acc),
    "voice_accuracy": float(b_voice_acc),
    "mean_cpu_latency_ms": float(b_latency_ms),
}

print(f"\nSelected Baseline: C={best_c} | Test Acc={b_acc:.4f} | Macro-F1={b_f1:.4f} | CPU={b_latency_ms:.3f} ms/query")
print("\n--- Baseline Classification Report (Test Split) ---")
print(classification_report(test_df["label"], baseline_test_preds, target_names=LABELS, digits=4))

## 4. Fine-Tune `distilbert-base-uncased` (Tier 1 Router)

In [ ]:
import inspect
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(SEED)
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class IntentDataset(Dataset):
    def __init__(self, texts, labels, tok, max_len=64):
        self.encodings = tok(
            list(texts),
            truncation=True,
            padding="max_length",
            max_length=max_len,
            return_tensors="pt",
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


train_ds = IntentDataset(train_df["text"], train_df["label"], tokenizer, MAX_LENGTH)
val_ds = IntentDataset(val_df["text"], val_df["label"], tokenizer, MAX_LENGTH)
test_ds = IntentDataset(test_df["text"], test_df["label"], tokenizer, MAX_LENGTH)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {
        "accuracy": float(acc),
        "macro_precision": float(prec),
        "macro_recall": float(rec),
        "macro_f1": float(f1),
    }


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

ckpt_dir = OUT_ROOT / "checkpoints"
ta_params = inspect.signature(TrainingArguments.__init__).parameters
strategy_key = "eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"

ta_kwargs = {
    "output_dir": str(ckpt_dir),
    "num_train_epochs": 6,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 32,
    "learning_rate": 3e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "logging_strategy": "epoch",
    strategy_key: "epoch",
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "macro_f1",
    "greater_is_better": True,
    "save_total_limit": 2,
    "fp16": torch.cuda.is_available(),
    "seed": SEED,
    "report_to": "none",
}
training_args = TrainingArguments(**ta_kwargs)

trainer_params = inspect.signature(Trainer.__init__).parameters
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_ds,
    "eval_dataset": val_ds,
    "compute_metrics": compute_metrics,
}
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)
train_result = trainer.train()

# Extract per-epoch history cleanly
epoch_rows = {}
for entry in trainer.state.log_history:
    if "epoch" not in entry:
        continue
    ep = int(round(entry["epoch"]))
    row = epoch_rows.setdefault(ep, {"epoch": ep})
    if "loss" in entry:
        row["train_loss"] = float(entry["loss"])
    if "eval_loss" in entry:
        row["eval_loss"] = float(entry["eval_loss"])
        row["eval_accuracy"] = float(entry.get("eval_accuracy", 0.0))
        row["eval_macro_f1"] = float(entry.get("eval_macro_f1", 0.0))

history_df = pd.DataFrame(sorted(epoch_rows.values(), key=lambda x: x["epoch"]))
print("\nPer-Epoch Training History:")
display(history_df)

## 5. Training vs. Validation Loss Curve (`loss_curve.png`)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), dpi=300)

best_row = history_df.loc[history_df["eval_macro_f1"].idxmax()]
best_ep = int(best_row["epoch"])

axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", linewidth=2.2, label="Training Loss")
axes[0].plot(history_df["epoch"], history_df["eval_loss"], marker="s", linewidth=2.2, label="Validation Loss")
axes[0].axvline(best_ep, color="gray", linestyle="--", alpha=0.7, label=f"Best Checkpoint (Epoch {best_ep})")
axes[0].set_title("DistilBERT Training vs. Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].set_xticks(history_df["epoch"])
axes[0].grid(True, alpha=0.25)
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["eval_accuracy"], marker="o", linewidth=2.2, label="Validation Accuracy")
axes[1].plot(history_df["epoch"], history_df["eval_macro_f1"], marker="^", linewidth=2.2, label="Validation Macro-F1")
axes[1].axvline(best_ep, color="gray", linestyle="--", alpha=0.7, label=f"Best Checkpoint (Epoch {best_ep})")
axes[1].set_title("Validation Accuracy & Macro-F1 by Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_ylim(0.5, 1.02)
axes[1].set_xticks(history_df["epoch"])
axes[1].grid(True, alpha=0.25)
axes[1].legend()

plt.tight_layout()
loss_png = CHARTS_DIR / "loss_curve.png"
plt.savefig(loss_png, dpi=300, bbox_inches="tight")
plt.show()
print(f"[OK] Saved {loss_png}")

## 6. DistilBERT Per-Class Report, Confusion Matrix & Baseline Comparison Table

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

val_pred_out = trainer.predict(val_ds)
test_pred_out = trainer.predict(test_ds)

val_logits = val_pred_out.predictions
val_labels = val_pred_out.label_ids
test_logits = test_pred_out.predictions
test_labels = test_pred_out.label_ids

d_preds = np.argmax(test_logits, axis=-1)
d_acc = accuracy_score(test_labels, d_preds)
d_prec, d_rec, d_f1, _ = precision_recall_fscore_support(
    test_labels, d_preds, average="macro", zero_division=0
)
_, _, d_wf1, _ = precision_recall_fscore_support(
    test_labels, d_preds, average="weighted", zero_division=0
)
d_typed_acc = accuracy_score(test_labels[typed_mask], d_preds[typed_mask])
d_voice_acc = accuracy_score(test_labels[voice_mask], d_preds[voice_mask])

distilbert_report_dict = classification_report(
    test_labels, d_preds, target_names=LABELS, digits=4, output_dict=True
)
print("--- DistilBERT Classification Report (Test Split) ---")
print(classification_report(test_labels, d_preds, target_names=LABELS, digits=4))

comparison_df = pd.DataFrame([
    {
        "Model": baseline_metrics["model"],
        "Accuracy": round(baseline_metrics["test_accuracy"], 4),
        "Macro_Precision": round(baseline_metrics["macro_precision"], 4),
        "Macro_Recall": round(baseline_metrics["macro_recall"], 4),
        "Macro_F1": round(baseline_metrics["macro_f1"], 4),
        "Weighted_F1": round(baseline_metrics["weighted_f1"], 4),
        "Typed_Acc": round(baseline_metrics["typed_accuracy"], 4),
        "Voice_Acc": round(baseline_metrics["voice_accuracy"], 4),
    },
    {
        "Model": "Fine-Tuned DistilBERT (FP32)",
        "Accuracy": round(float(d_acc), 4),
        "Macro_Precision": round(float(d_prec), 4),
        "Macro_Recall": round(float(d_rec), 4),
        "Macro_F1": round(float(d_f1), 4),
        "Weighted_F1": round(float(d_wf1), 4),
        "Typed_Acc": round(float(d_typed_acc), 4),
        "Voice_Acc": round(float(d_voice_acc), 4),
    },
])
delta_row = {
    "Model": "Delta (DistilBERT - Baseline)",
    "Accuracy": round(float(d_acc - baseline_metrics["test_accuracy"]), 4),
    "Macro_Precision": round(float(d_prec - baseline_metrics["macro_precision"]), 4),
    "Macro_Recall": round(float(d_rec - baseline_metrics["macro_recall"]), 4),
    "Macro_F1": round(float(d_f1 - baseline_metrics["macro_f1"]), 4),
    "Weighted_F1": round(float(d_wf1 - baseline_metrics["weighted_f1"]), 4),
    "Typed_Acc": round(float(d_typed_acc - baseline_metrics["typed_accuracy"]), 4),
    "Voice_Acc": round(float(d_voice_acc - baseline_metrics["voice_accuracy"]), 4),
}
comparison_df = pd.concat([comparison_df, pd.DataFrame([delta_row])], ignore_index=True)
comparison_csv = CHARTS_DIR / "baseline_vs_distilbert.csv"
comparison_df.to_csv(comparison_csv, index=False)
print("\n--- Baseline vs. DistilBERT Comparison (Identical 80/10/10 Split) ---")
display(comparison_df)

cm = confusion_matrix(test_labels, d_preds)
fig, ax = plt.subplots(figsize=(9.5, 8.2), dpi=300)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS)
disp.plot(ax=ax, cmap="Blues", xticks_rotation=35, values_format="d", colorbar=False)
ax.set_title("J.U.D.E. Tier 1 DistilBERT — Test Set Confusion Matrix (n=108)")
plt.tight_layout()
cm_png = CHARTS_DIR / "confusion_matrix.png"
plt.savefig(cm_png, dpi=300, bbox_inches="tight")
plt.show()

# Error analysis
test_probs_raw = torch.softmax(torch.tensor(test_logits), dim=-1).numpy()
err_indices = np.where(d_preds != test_labels)[0]
print(f"\nMisclassified Test Utterances ({len(err_indices)} / {len(test_labels)}):")
for idx in err_indices:
    print(
        f"  - [{test_df.loc[idx, 'style']}] {test_df.loc[idx, 'text']!r} | "
        f"True={id2label[int(test_labels[idx])]} | Pred={id2label[int(d_preds[idx])]} "
        f"(conf={test_probs_raw[idx].max():.3f})"
    )

## 7. Temperature Scaling Calibration & OOD 3-Way Confidence Gate Validation

In [ ]:
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score

HIGH_THRESH = 0.80
LOW_THRESH = 0.55


def softmax_np(logits, temperature=1.0):
    scaled = logits / temperature
    scaled = scaled - np.max(scaled, axis=-1, keepdims=True)
    exp_x = np.exp(scaled)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)


def nll_score(logits, labels, temperature=1.0):
    probs = softmax_np(logits, temperature)
    p_true = np.clip(probs[np.arange(len(labels)), labels], 1e-12, 1.0)
    return float(-np.mean(np.log(p_true)))


def expected_calibration_error(probs, labels, n_bins=10):
    confs = np.max(probs, axis=-1)
    preds = np.argmax(probs, axis=-1)
    accs = (preds == labels).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for b_lo, b_hi in zip(bins[:-1], bins[1:]):
        if b_hi == 1.0:
            mask = (confs >= b_lo) & (confs <= b_hi)
        else:
            mask = (confs >= b_lo) & (confs < b_hi)
        if np.any(mask):
            ece += (np.sum(mask) / len(labels)) * abs(np.mean(accs[mask]) - np.mean(confs[mask]))
    return float(ece)


# Fit temperature T on validation logits ONLY (no test leakage)
res = minimize(
    lambda t: nll_score(val_logits, val_labels, float(t[0])),
    x0=[1.0],
    bounds=[(0.2, 5.0)],
    method="L-BFGS-B",
)
t_opt = float(res.x[0])

val_nll_before = nll_score(val_logits, val_labels, 1.0)
val_nll_after = nll_score(val_logits, val_labels, t_opt)
temp_applied = val_nll_after < val_nll_before
T_FINAL = t_opt if temp_applied else 1.0

val_ece_before = expected_calibration_error(softmax_np(val_logits, 1.0), val_labels)
val_ece_after = expected_calibration_error(softmax_np(val_logits, T_FINAL), val_labels)

test_probs_before = softmax_np(test_logits, 1.0)
test_probs_after = softmax_np(test_logits, T_FINAL)
test_nll_before = nll_score(test_logits, test_labels, 1.0)
test_nll_after = nll_score(test_logits, test_labels, T_FINAL)
test_ece_before = expected_calibration_error(test_probs_before, test_labels)
test_ece_after = expected_calibration_error(test_probs_after, test_labels)

print(f"Temperature Scaling Optimization: T_opt={t_opt:.4f} | Applied={temp_applied} -> T_FINAL={T_FINAL:.4f}")
print(f"Validation -> NLL: {val_nll_before:.4f} -> {val_nll_after:.4f} | ECE: {val_ece_before:.4f} -> {val_ece_after:.4f}")
print(f"Test       -> NLL: {test_nll_before:.4f} -> {test_nll_after:.4f} | ECE: {test_ece_before:.4f} -> {test_ece_after:.4f}")

# Predict on OOD utterances
ood_ds = IntentDataset(ood_df["text"], np.zeros(len(ood_df), dtype=int), tokenizer, MAX_LENGTH)
ood_logits = trainer.predict(ood_ds).predictions
ood_probs_before = softmax_np(ood_logits, 1.0)
ood_probs_after = softmax_np(ood_logits, T_FINAL)

id_confs_before = np.max(test_probs_before, axis=-1)
id_confs_after = np.max(test_probs_after, axis=-1)
ood_confs_before = np.max(ood_probs_before, axis=-1)
ood_confs_after = np.max(ood_probs_after, axis=-1)


def gate_bucket(conf):
    if conf >= HIGH_THRESH:
        return "Direct (>=0.80)"
    if conf >= LOW_THRESH:
        return "Confirm (0.55-0.80)"
    return "Escalate (<0.55)"


def summarize_gate(confs, name):
    n = len(confs)
    hi = int(np.sum(confs >= HIGH_THRESH))
    mid = int(np.sum((confs >= LOW_THRESH) & (confs < HIGH_THRESH)))
    lo = int(np.sum(confs < LOW_THRESH))
    return {
        "Split": name,
        "Count": n,
        "Direct_>=0.80": f"{hi} ({100*hi/n:.1f}%)",
        "Confirm_0.55-0.80": f"{mid} ({100*mid/n:.1f}%)",
        "Escalate_<0.55": f"{lo} ({100*lo/n:.1f}%)",
        "Mean_Conf": round(float(np.mean(confs)), 4),
    }


gate_summary_df = pd.DataFrame([
    summarize_gate(id_confs_before, "ID Test (Uncalibrated T=1.0)"),
    summarize_gate(ood_confs_before, "OOD Test (Uncalibrated T=1.0)"),
    summarize_gate(id_confs_after, f"ID Test (Calibrated T={T_FINAL:.2f})"),
    summarize_gate(ood_confs_after, f"OOD Test (Calibrated T={T_FINAL:.2f})"),
])
print("\n--- 3-Way Confidence Gate Summary ---")
display(gate_summary_df)

id_direct_mask = id_confs_after >= HIGH_THRESH
id_direct_acc = accuracy_score(test_labels[id_direct_mask], d_preds[id_direct_mask]) if np.any(id_direct_mask) else 0.0
ood_caught_below_high = float(np.mean(ood_confs_after < HIGH_THRESH))
ood_escalated_below_low = float(np.mean(ood_confs_after < LOW_THRESH))
ood_roc_auc = float(roc_auc_score(
    np.concatenate([np.ones(len(id_confs_after)), np.zeros(len(ood_confs_after))]),
    np.concatenate([id_confs_after, ood_confs_after]),
))

print(f"ID Direct-Route Accuracy (>=0.80) : {id_direct_acc:.4f} ({int(np.sum(id_direct_mask))}/{len(id_confs_after)} queries)")
print(f"OOD Blocked from Direct (<0.80)   : {ood_caught_below_high*100:.1f}%")
print(f"OOD Auto-Escalated (<0.55)        : {ood_escalated_below_low*100:.1f}%")
print(f"ID vs. OOD ROC-AUC (Max Softmax)  : {ood_roc_auc:.4f}")

ood_out_df = ood_df.copy()
ood_pred_ids = np.argmax(ood_probs_after, axis=-1)
ood_out_df["predicted_intent"] = [id2label[int(i)] for i in ood_pred_ids]
ood_out_df["conf_uncalibrated"] = np.round(ood_confs_before, 4)
ood_out_df["conf_calibrated"] = np.round(ood_confs_after, 4)
ood_out_df["gate_action"] = [gate_bucket(c) for c in ood_confs_after]
ood_csv = CHARTS_DIR / "ood_predictions.csv"
ood_out_df.to_csv(ood_csv, index=False)
display(ood_out_df.head(15))

# Confidence Histogram PNG (Uncalibrated vs Calibrated)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), dpi=300, sharey=True)
bins = np.linspace(0.0, 1.0, 21)

for ax, id_c, ood_c, title in [
    (axes[0], id_confs_before, ood_confs_before, "Uncalibrated Softmax (T = 1.00)"),
    (axes[1], id_confs_after, ood_confs_after, f"Temperature-Scaled Softmax (T = {T_FINAL:.2f})"),
]:
    ax.hist(id_c, bins=bins, density=True, alpha=0.65, label=f"In-Distribution Test (n={len(id_c)})")
    ax.hist(ood_c, bins=bins, density=True, alpha=0.65, label=f"OOD / Ambiguous (n={len(ood_c)})")
    ax.axvline(LOW_THRESH, color="crimson", linestyle="--", linewidth=1.8, label="Escalate Gate (0.55)")
    ax.axvline(HIGH_THRESH, color="darkgreen", linestyle="--", linewidth=1.8, label="Direct Gate (0.80)")
    ax.set_title(title)
    ax.set_xlabel("Max Predicted Softmax Confidence")
    ax.set_xlim(0.0, 1.02)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper left", fontsize=8.5)

axes[0].set_ylabel("Density")
fig.suptitle("J.U.D.E. Tier 1 Confidence Calibration & 3-Way Gate Separation", y=1.02, fontsize=12)
plt.tight_layout()
conf_png = CHARTS_DIR / "confidence_histogram_ood.png"
plt.savefig(conf_png, dpi=300, bbox_inches="tight")
plt.show()
print(f"[OK] Saved {conf_png}")

## 8. Export FP32 ONNX, Quantize to INT8 ONNX & Run CPU Benchmark

In [ ]:
import shutil
from onnxruntime.quantization import QuantType, quantize_dynamic
from tokenizers import Tokenizer as FastTokenizer

cpu_model = trainer.model.to("cpu").eval()

# Measure FP32 PyTorch weights size on disk
pt_tmp_dir = OUT_ROOT / "pt_fp32_tmp"
if pt_tmp_dir.exists():
    shutil.rmtree(pt_tmp_dir)
cpu_model.save_pretrained(pt_tmp_dir)
pt_weight_file = next(
    p for p in pt_tmp_dir.iterdir()
    if p.name in ("model.safetensors", "pytorch_model.bin")
)
pytorch_fp32_mb = pt_weight_file.stat().st_size / (1024 * 1024)

# Export FP32 ONNX
fp32_onnx_path = OUT_ROOT / "model_fp32.onnx"
quant_onnx_path = ROUTER_DIR / "model_quantized.onnx"

dummy = tokenizer(
    "compress my semester notes folder into a zip",
    return_tensors="pt",
    max_length=MAX_LENGTH,
    padding="max_length",
    truncation=True,
)
export_kwargs = {
    "f": str(fp32_onnx_path),
    "input_names": ["input_ids", "attention_mask"],
    "output_names": ["logits"],
    "dynamic_axes": {
        "input_ids": {0: "batch_size", 1: "seq_len"},
        "attention_mask": {0: "batch_size", 1: "seq_len"},
        "logits": {0: "batch_size"},
    },
    "opset_version": 14,
}
if "dynamo" in inspect.signature(torch.onnx.export).parameters:
    export_kwargs["dynamo"] = False

with torch.no_grad():
    torch.onnx.export(
        cpu_model,
        (dummy["input_ids"], dummy["attention_mask"]),
        **export_kwargs,
    )

onnx_fp32_mb = fp32_onnx_path.stat().st_size / (1024 * 1024)

# Dynamic INT8 quantization
quantize_dynamic(
    model_input=str(fp32_onnx_path),
    model_output=str(quant_onnx_path),
    weight_type=QuantType.QUInt8,
    op_types_to_quantize=["MatMul", "Gemm"],
)
onnx_int8_mb = quant_onnx_path.stat().st_size / (1024 * 1024)

# Save tokenizer files & verify 100% parity with lightweight `tokenizers` package
tokenizer.save_pretrained(str(ROUTER_DIR))
fast_tok = FastTokenizer.from_file(str(ROUTER_DIR / "tokenizer.json"))
fast_tok.enable_truncation(max_length=MAX_LENGTH)

for txt in test_df["text"].iloc[:20]:
    hf_ids = tokenizer.encode(txt, truncation=True, max_length=MAX_LENGTH)
    fast_ids = fast_tok.encode(txt).ids
    assert hf_ids == fast_ids, f"Tokenizer mismatch on {txt!r}: {hf_ids} != {fast_ids}"
print("[OK] Verified 100% token-ID parity between Hugging Face AutoTokenizer and `tokenizers.Tokenizer`")

# Save label_map.json for local J.U.D.E. runtime
label_map_payload = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "temperature": round(float(T_FINAL), 6),
    "thresholds": {"high": HIGH_THRESH, "low": LOW_THRESH},
    "id2label": {str(k): v for k, v in id2label.items()},
    "label2id": label2id,
    "intent_to_tier": intent_to_tier,
}
with (ROUTER_DIR / "label_map.json").open("w", encoding="utf-8") as f:
    json.dump(label_map_payload, f, indent=2)

# CPU Latency & Accuracy Benchmark: FP32 PyTorch vs INT8 ONNX
bench_texts = test_df["text"].tolist()

# Warmup + timed FP32 PyTorch CPU
for txt in bench_texts[:10]:
    enc = tokenizer(txt, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    with torch.no_grad():
        _ = cpu_model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])

pt_latencies = []
pt_preds = []
for txt in bench_texts:
    t0 = time.perf_counter()
    enc = tokenizer(txt, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    with torch.no_grad():
        logits = cpu_model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits.numpy()
    pt_latencies.append((time.perf_counter() - t0) * 1000.0)
    pt_preds.append(int(np.argmax(logits, axis=-1)[0]))

# Warmup + timed INT8 ONNX CPU (exact local runtime stack: tokenizers + onnxruntime)
sess_opts = ort.SessionOptions()
sess_opts.intra_op_num_threads = 4
ort_session = ort.InferenceSession(
    str(quant_onnx_path), sess_options=sess_opts, providers=["CPUExecutionProvider"]
)

for txt in bench_texts[:10]:
    enc = fast_tok.encode(txt)
    ids = np.array([enc.ids], dtype=np.int64)
    mask = np.array([enc.attention_mask], dtype=np.int64)
    _ = ort_session.run(None, {"input_ids": ids, "attention_mask": mask})

ort_latencies = []
ort_preds = []
for txt in bench_texts:
    t0 = time.perf_counter()
    enc = fast_tok.encode(txt)
    ids = np.array([enc.ids], dtype=np.int64)
    mask = np.array([enc.attention_mask], dtype=np.int64)
    logits = ort_session.run(None, {"input_ids": ids, "attention_mask": mask})[0]
    ort_latencies.append((time.perf_counter() - t0) * 1000.0)
    ort_preds.append(int(np.argmax(logits, axis=-1)[0]))

pt_acc = accuracy_score(test_labels, pt_preds)
_, _, pt_f1, _ = precision_recall_fscore_support(test_labels, pt_preds, average="macro", zero_division=0)
ort_acc = accuracy_score(test_labels, ort_preds)
_, _, ort_f1, _ = precision_recall_fscore_support(test_labels, ort_preds, average="macro", zero_division=0)

bench_df = pd.DataFrame([
    {
        "Runtime": "PyTorch FP32 (CPU)",
        "Size_MB": round(pytorch_fp32_mb, 2),
        "Mean_ms": round(float(np.mean(pt_latencies)), 2),
        "P50_ms": round(float(np.percentile(pt_latencies, 50)), 2),
        "P95_ms": round(float(np.percentile(pt_latencies, 95)), 2),
        "Test_Accuracy": round(float(pt_acc), 4),
        "Test_Macro_F1": round(float(pt_f1), 4),
    },
    {
        "Runtime": "ONNX INT8 (CPU)",
        "Size_MB": round(onnx_int8_mb, 2),
        "Mean_ms": round(float(np.mean(ort_latencies)), 2),
        "P50_ms": round(float(np.percentile(ort_latencies, 50)), 2),
        "P95_ms": round(float(np.percentile(ort_latencies, 95)), 2),
        "Test_Accuracy": round(float(ort_acc), 4),
        "Test_Macro_F1": round(float(ort_f1), 4),
    },
])
size_reduction_pct = (1.0 - onnx_int8_mb / pytorch_fp32_mb) * 100.0
speedup_x = float(np.mean(pt_latencies) / np.mean(ort_latencies))
print(f"\n--- FP32 PyTorch vs. INT8 ONNX Benchmark (Size -{size_reduction_pct:.1f}% | Speedup {speedup_x:.2f}x) ---")
display(bench_df)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4), dpi=300)
runtimes = bench_df["Runtime"].tolist()
axes[0].bar(runtimes, bench_df["Size_MB"], color=["#4C78A8", "#54A24B"])
axes[0].set_title(f"Model Footprint (-{size_reduction_pct:.1f}% Size)")
axes[0].set_ylabel("Size on Disk (MB)")
axes[0].grid(axis="y", alpha=0.25)

axes[1].bar(runtimes, bench_df["Mean_ms"], color=["#4C78A8", "#54A24B"])
axes[1].set_title(f"Single-Query CPU Latency ({speedup_x:.2f}x Speedup)")
axes[1].set_ylabel("Mean Latency (ms)")
axes[1].grid(axis="y", alpha=0.25)

plt.tight_layout()
bench_png = CHARTS_DIR / "fp32_vs_int8_benchmark.png"
plt.savefig(bench_png, dpi=300, bbox_inches="tight")
plt.show()
print(f"[OK] Saved {bench_png}")

## 9. Save `metrics.json`, Package Bundles & Download

In [ ]:
import zipfile

metrics_payload = {
    "dataset": {
        "total_in_distribution": int(len(df)),
        "train_count": int(len(train_df)),
        "val_count": int(len(val_df)),
        "test_count": int(len(test_df)),
        "ood_count": int(len(ood_df)),
        "intents": LABELS,
        "seed": SEED,
    },
    "baseline": baseline_metrics,
    "distilbert_fp32": {
        "model_name": MODEL_NAME,
        "epochs": 6,
        "best_epoch": int(best_ep),
        "test_accuracy": float(d_acc),
        "macro_precision": float(d_prec),
        "macro_recall": float(d_rec),
        "macro_f1": float(d_f1),
        "weighted_f1": float(d_wf1),
        "typed_accuracy": float(d_typed_acc),
        "voice_accuracy": float(d_voice_acc),
        "per_class_report": distilbert_report_dict,
    },
    "calibration_and_ood": {
        "temperature_opt": float(t_opt),
        "temperature_applied": bool(temp_applied),
        "temperature_final": float(T_FINAL),
        "thresholds": {"high": HIGH_THRESH, "low": LOW_THRESH},
        "val_nll_before": float(val_nll_before),
        "val_nll_after": float(val_nll_after),
        "val_ece_before": float(val_ece_before),
        "val_ece_after": float(val_ece_after),
        "test_nll_before": float(test_nll_before),
        "test_nll_after": float(test_nll_after),
        "test_ece_before": float(test_ece_before),
        "test_ece_after": float(test_ece_after),
        "id_direct_accuracy": float(id_direct_acc),
        "ood_blocked_below_0_80": float(ood_caught_below_high),
        "ood_escalated_below_0_55": float(ood_escalated_below_low),
        "id_vs_ood_roc_auc": float(ood_roc_auc),
    },
    "onnx_benchmark": {
        "pytorch_fp32_mb": round(pytorch_fp32_mb, 2),
        "onnx_fp32_mb": round(onnx_fp32_mb, 2),
        "onnx_int8_mb": round(onnx_int8_mb, 2),
        "size_reduction_pct": round(size_reduction_pct, 2),
        "speedup_x": round(speedup_x, 2),
        "pytorch_cpu_ms": {
            "mean": round(float(np.mean(pt_latencies)), 2),
            "p50": round(float(np.percentile(pt_latencies, 50)), 2),
            "p95": round(float(np.percentile(pt_latencies, 95)), 2),
        },
        "onnx_int8_cpu_ms": {
            "mean": round(float(np.mean(ort_latencies)), 2),
            "p50": round(float(np.percentile(ort_latencies, 50)), 2),
            "p95": round(float(np.percentile(ort_latencies, 95)), 2),
        },
        "onnx_int8_test_accuracy": float(ort_acc),
        "onnx_int8_test_macro_f1": float(ort_f1),
    },
}

for target in (ROUTER_DIR / "metrics.json", CHARTS_DIR / "metrics.json"):
    with target.open("w", encoding="utf-8") as f:
        json.dump(metrics_payload, f, indent=2)

router_zip = OUT_ROOT / "tier1_router_bundle.zip"
router_files = [
    "model_quantized.onnx",
    "tokenizer.json",
    "tokenizer_config.json",
    "label_map.json",
    "metrics.json",
]
with zipfile.ZipFile(router_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in router_files:
        fpath = ROUTER_DIR / fname
        if fpath.exists():
            zf.write(fpath, arcname=fname)

report_zip = OUT_ROOT / "cl461_report_assets.zip"
with zipfile.ZipFile(report_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fpath in sorted(CHARTS_DIR.iterdir()):
        if fpath.is_file():
            zf.write(fpath, arcname=fpath.name)

print(f"[OK] Packaged {router_zip} ({router_zip.stat().st_size / (1024*1024):.2f} MB)")
print(f"[OK] Packaged {report_zip} ({report_zip.stat().st_size / (1024*1024):.2f} MB)")
print("Next: extract tier1_router_bundle.zip into models/tier1_router/ inside your J.U.D.E. folder.")

try:
    from google.colab import files
    files.download(str(router_zip))
    files.download(str(report_zip))
except ImportError:
    print("Not running inside Google Colab UI; skipping browser download.")

# Optional Google Drive backup helper:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/jude_outputs/*.zip /content/drive/MyDrive/